In [ ]:
# ============================================================
# CLAHE PREPROCESSING PIPELINE
# ============================================================
# Purpose : Apply Contrast Limited Adaptive Histogram
#           Equalization (CLAHE) to all aggregate images,
#           segment each particle from the background, and
#           replace the background with a uniform white fill.
# Input   : Cropped aggregate images per milling class
#           (Google Drive: BG_Dataset/<class>/)
# Output  : CLAHE-enhanced images with white background
#           (Google Drive: BG CLAHE/<class>/)
# Classes : 0, 100, 500, 1000, 1500, 2000 RPM
# Settings: clip limit = 2.0, tile grid = 8x8
# ============================================================
# Note    : Each cell below processes one milling class.
#           Run all cells sequentially to process the full
#           dataset. The pipeline is identical for each class.
# ============================================================

import cv2
import numpy as np
# ── Libraries ─────────────────────────────────────────────────
# cv2      : OpenCV for image processing (CLAHE, thresholding,
#            morphology, contour detection)
# numpy    : Array operations for mask creation and pixel ops
# os/glob  : File system traversal and path construction
# tqdm     : Terminal progress bar for batch processing
import os
from glob import glob
# Mount Google Drive to access input images and save output
from google.colab import drive
from tqdm import tqdm  # Import tqdm for progress bar

# Mount Google Drive
drive.mount('/content/drive')

# === Set the path to your already extracted BG Dataset ===

# ── Paths ──────────────────────────────────────────────────────
# class_folder : Input directory for this milling class
# output_folder: Destination for CLAHE-processed images
class_folder = '/content/drive/MyDrive/BG_Dataset/0'  # Path to class '0' folder
output_folder = '/content/drive/MyDrive/BG CLAHE/0'  # Save processed images to BG_CLAHE folder
os.makedirs(output_folder, exist_ok=True)

# === Process all images in class '0' ===

# Discover all image files in the class folder (sorted for
# reproducibility) and wrap in a tqdm progress bar
image_paths = sorted(glob(os.path.join(class_folder, '*')))
print(f"Found {len(image_paths)} images in class '0'.")

# Initialize tqdm progress bar
with tqdm(total=len(image_paths), desc="Processing images", unit="image") as pbar:
    for path in image_paths:

        # ── Per-image pipeline ─────────────────────────────────────────
        filename = os.path.basename(path)
        raw_img = cv2.imread(path)
        # Skip unreadable files (corrupt or unsupported format)
        if raw_img is None:
            print(f"Skipped: {filename}")
            pbar.update(1)
            continue


        # ── Step 1: Grayscale conversion ───────────────────────────────
        # Convert BGR image to grayscale for single-channel processing.
        # CLAHE operates on 2-D intensity images, not colour images.

        # ── Step 2: Apply CLAHE ────────────────────────────────────────
        # clipLimit=2.0  : Maximum contrast amplification per tile.
        #                  Values above this are clipped and redistributed,
        #                  preventing noise amplification.
        # tileGridSize=(8,8): Divides the image into 64 local tiles.
        #                  Each tile gets its own histogram equalization,
        #                  enabling local contrast enhancement.
        # Bilinear interpolation smooths tile boundaries on merge.
        # Grayscale + CLAHE
        gray = cv2.cvtColor(raw_img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        clahe_img = clahe.apply(gray)


        # ── Step 3: Binary thresholding ────────────────────────────────
        # THRESH_BINARY_INV: pixels below threshold=120 become white
        # (aggregate foreground), above become black (background).
        # Threshold of 120 separates the darker aggregate particle
        # from the lighter white background used during imaging.

        # ── Step 4: Morphological opening ──────────────────────────────
        # Erode then dilate with a 3x3 kernel (2 iterations) to remove
        # small isolated noise regions while preserving the main
        # aggregate contour shape.
        # Threshold + cleanup
        _, mask = cv2.threshold(clahe_img, 120, 255, cv2.THRESH_BINARY_INV)
        kernel = np.ones((3, 3), np.uint8)
        clean_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)


        # ── Step 5: Largest contour extraction ─────────────────────────
        # RETR_EXTERNAL retrieves only outermost contours (no holes).
        # Selecting the largest contour by area isolates the single
        # aggregate particle and ignores all background noise blobs.
        # Find largest contour
        contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        # Skip images where no particle contour could be detected
        if not contours:
            print(f"No contour in: {filename}")
            pbar.update(1)
            continue
        # Identify the largest contour — the aggregate particle boundary
        largest = max(contours, key=cv2.contourArea)
        # Create a binary mask by filling the largest contour solid white
        mask_fill = np.zeros_like(gray)
        cv2.drawContours(mask_fill, [largest], -1, 255, thickness=-1)


        # ── Step 6: White background replacement ───────────────────────
        # Pixels inside the particle mask retain their CLAHE-enhanced
        # intensity values. All background pixels are set to 255 (white),
        # producing a clean, consistent background across all images.
        # White background
        white_bg = np.full_like(clahe_img, 255)
        result = np.where(mask_fill == 255, clahe_img, white_bg)


        # ── Step 7: Save processed image ───────────────────────────────
        # Write the CLAHE-enhanced, white-background image to the output
        # folder, preserving the original filename for traceability.
        # Save result
        out_path = os.path.join(output_folder, filename)
        cv2.imwrite(out_path, result)
        print(f"Saved: {filename}")

        # Advance tqdm progress bar by one image
        # Update progress bar
        pbar.update(1)

print(f"\nDone! Processed images from class '0' saved in: {output_folder}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 Found 4546 images in class '0'.


Processing images:   0%|          | 1/4546 [00:00<21:02,  3.60image/s]

✅ Saved: 10_01.png


Processing images:   0%|          | 3/4546 [00:00<17:13,  4.40image/s]

✅ Saved: 10_02.png
✅ Saved: 10_03.png


...

✅ Saved: 9_45.png
✅ Saved: 9_46.png
✅ Saved: 9_47.png
✅ Saved: 9_48.png
✅ Saved: 9_49.png
✅ Saved: 9_50.png
✅ Saved: 9_51.png
✅ Saved: 9_52.png
✅ Saved: 9_53.png
✅ Saved: 9_54.png
✅ Saved: 9_55.png


Processing images: 100%|█████████▉| 4540/4546 [02:35<00:00, 56.09image/s]

✅ Saved: 9_56.png
✅ Saved: 9_57.png
✅ Saved: 9_58.png
✅ Saved: 9_59.png
✅ Saved: 9_60.png
✅ Saved: 9_61.png
✅ Saved: 9_62.png
✅ Saved: 9_63.png
✅ Saved: 9_64.png
✅ Saved: 9_65.png
✅ Saved: 9_66.png
✅ Saved: 9_67.png


Processing images: 100%|██████████| 4546/4546 [02:35<00:00, 29.29image/s]

✅ Saved: 9_68.png
✅ Saved: 9_69.png
✅ Saved: 9_70.png

🎉 Done! Processed images from class '0' saved in: /content/drive/MyDrive/BG CLAHE/0


In [ ]:
import cv2
import numpy as np
# ── Libraries ─────────────────────────────────────────────────
# cv2      : OpenCV for image processing (CLAHE, thresholding,
#            morphology, contour detection)
# numpy    : Array operations for mask creation and pixel ops
# os/glob  : File system traversal and path construction
# tqdm     : Terminal progress bar for batch processing
import os
from glob import glob
# Mount Google Drive to access input images and save output
from google.colab import drive
from tqdm import tqdm  # Import tqdm for progress bar

# Mount Google Drive
drive.mount('/content/drive')

# === Set the path to your already extracted BG Dataset ===

# ── Paths ──────────────────────────────────────────────────────
# class_folder : Input directory for this milling class
# output_folder: Destination for CLAHE-processed images
class_folder = '/content/drive/MyDrive/BG_Dataset/100'  # Path to class '0' folder
output_folder = '/content/drive/MyDrive/BG CLAHE/100'  # Save processed images to BG_CLAHE folder
os.makedirs(output_folder, exist_ok=True)

# === Process all images in class '0' ===

# Discover all image files in the class folder (sorted for
# reproducibility) and wrap in a tqdm progress bar
image_paths = sorted(glob(os.path.join(class_folder, '*')))
print(f"Found {len(image_paths)} images in class '100'.")

# Initialize tqdm progress bar
with tqdm(total=len(image_paths), desc="Processing images", unit="image") as pbar:
    for path in image_paths:

        # ── Per-image pipeline ─────────────────────────────────────────
        filename = os.path.basename(path)
        raw_img = cv2.imread(path)
        # Skip unreadable files (corrupt or unsupported format)
        if raw_img is None:
            print(f"Skipped: {filename}")
            pbar.update(1)
            continue


        # ── Step 1: Grayscale conversion ───────────────────────────────
        # Convert BGR image to grayscale for single-channel processing.
        # CLAHE operates on 2-D intensity images, not colour images.

        # ── Step 2: Apply CLAHE ────────────────────────────────────────
        # clipLimit=2.0  : Maximum contrast amplification per tile.
        #                  Values above this are clipped and redistributed,
        #                  preventing noise amplification.
        # tileGridSize=(8,8): Divides the image into 64 local tiles.
        #                  Each tile gets its own histogram equalization,
        #                  enabling local contrast enhancement.
        # Bilinear interpolation smooths tile boundaries on merge.
        # Grayscale + CLAHE
        gray = cv2.cvtColor(raw_img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        clahe_img = clahe.apply(gray)


        # ── Step 3: Binary thresholding ────────────────────────────────
        # THRESH_BINARY_INV: pixels below threshold=120 become white
        # (aggregate foreground), above become black (background).
        # Threshold of 120 separates the darker aggregate particle
        # from the lighter white background used during imaging.

        # ── Step 4: Morphological opening ──────────────────────────────
        # Erode then dilate with a 3x3 kernel (2 iterations) to remove
        # small isolated noise regions while preserving the main
        # aggregate contour shape.
        # Threshold + cleanup
        _, mask = cv2.threshold(clahe_img, 120, 255, cv2.THRESH_BINARY_INV)
        kernel = np.ones((3, 3), np.uint8)
        clean_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)


        # ── Step 5: Largest contour extraction ─────────────────────────
        # RETR_EXTERNAL retrieves only outermost contours (no holes).
        # Selecting the largest contour by area isolates the single
        # aggregate particle and ignores all background noise blobs.
        # Find largest contour
        contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        # Skip images where no particle contour could be detected
        if not contours:
            print(f"No contour in: {filename}")
            pbar.update(1)
            continue
        # Identify the largest contour — the aggregate particle boundary
        largest = max(contours, key=cv2.contourArea)
        # Create a binary mask by filling the largest contour solid white
        mask_fill = np.zeros_like(gray)
        cv2.drawContours(mask_fill, [largest], -1, 255, thickness=-1)


        # ── Step 6: White background replacement ───────────────────────
        # Pixels inside the particle mask retain their CLAHE-enhanced
        # intensity values. All background pixels are set to 255 (white),
        # producing a clean, consistent background across all images.
        # White background
        white_bg = np.full_like(clahe_img, 255)
        result = np.where(mask_fill == 255, clahe_img, white_bg)


        # ── Step 7: Save processed image ───────────────────────────────
        # Write the CLAHE-enhanced, white-background image to the output
        # folder, preserving the original filename for traceability.
        # Save result
        out_path = os.path.join(output_folder, filename)
        cv2.imwrite(out_path, result)
        print(f"Saved: {filename}")

        # Advance tqdm progress bar by one image
        # Update progress bar
        pbar.update(1)

print(f"\nDone! Processed images from class '100' saved in: {output_folder}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 Found 4194 images in class '100'.


Processing images:   0%|          | 1/4194 [00:00<18:06,  3.86image/s]

✅ Saved: 100 Rev (1)_01.png


Processing images:   0%|          | 2/4194 [00:00<16:49,  4.15image/s]

✅ Saved: 100 Rev (1)_02.png


...

✅ Saved: 100 Rev (9)_42.png
✅ Saved: 100 Rev (9)_43.png
✅ Saved: 100 Rev (9)_44.png
✅ Saved: 100 Rev (9)_45.png
✅ Saved: 100 Rev (9)_46.png
✅ Saved: 100 Rev (9)_47.png
✅ Saved: 100 Rev (9)_48.png
✅ Saved: 100 Rev (9)_49.png
✅ Saved: 100 Rev (9)_50.png
✅ Saved: 100 Rev (9)_51.png
✅ Saved: 100 Rev (9)_52.png


Processing images: 100%|█████████▉| 4187/4194 [02:53<00:00, 56.84image/s]

✅ Saved: 100 Rev (9)_53.png
✅ Saved: 100 Rev (9)_54.png
✅ Saved: 100 Rev (9)_55.png
✅ Saved: 100 Rev (9)_56.png
✅ Saved: 100 Rev (9)_57.png
✅ Saved: 100 Rev (9)_58.png
✅ Saved: 100 Rev (9)_59.png
✅ Saved: 100 Rev (9)_60.png
✅ Saved: 100 Rev (9)_61.png
✅ Saved: 100 Rev (9)_62.png
✅ Saved: 100 Rev (9)_63.png
✅ Saved: 100 Rev (9)_64.png


Processing images: 100%|██████████| 4194/4194 [02:53<00:00, 24.14image/s]

✅ Saved: 100 Rev (9)_65.png
✅ Saved: 100 Rev (9)_66.png
✅ Saved: 100 Rev (9)_67.png
✅ Saved: 100 Rev (9)_68.png
✅ Saved: 100 Rev (9)_69.png
✅ Saved: 100 Rev (9)_70.png

🎉 Done! Processed images from class '100' saved in: /content/drive/MyDrive/BG CLAHE/100


In [ ]:
import cv2
import numpy as np
# ── Libraries ─────────────────────────────────────────────────
# cv2      : OpenCV for image processing (CLAHE, thresholding,
#            morphology, contour detection)
# numpy    : Array operations for mask creation and pixel ops
# os/glob  : File system traversal and path construction
# tqdm     : Terminal progress bar for batch processing
import os
from glob import glob
# Mount Google Drive to access input images and save output
from google.colab import drive
from tqdm import tqdm  # Import tqdm for progress bar

# Mount Google Drive
drive.mount('/content/drive')

# === Set the path to your already extracted BG Dataset ===

# ── Paths ──────────────────────────────────────────────────────
# class_folder : Input directory for this milling class
# output_folder: Destination for CLAHE-processed images
class_folder = '/content/drive/MyDrive/BG_Dataset/500'  # Path to class '0' folder
output_folder = '/content/drive/MyDrive/BG CLAHE/500'  # Save processed images to BG_CLAHE folder
os.makedirs(output_folder, exist_ok=True)

# === Process all images in class '0' ===

# Discover all image files in the class folder (sorted for
# reproducibility) and wrap in a tqdm progress bar
image_paths = sorted(glob(os.path.join(class_folder, '*')))
print(f"Found {len(image_paths)} images in class '500'.")

# Initialize tqdm progress bar
with tqdm(total=len(image_paths), desc="Processing images", unit="image") as pbar:
    for path in image_paths:

        # ── Per-image pipeline ─────────────────────────────────────────
        filename = os.path.basename(path)
        raw_img = cv2.imread(path)
        # Skip unreadable files (corrupt or unsupported format)
        if raw_img is None:
            print(f"Skipped: {filename}")
            pbar.update(1)
            continue


        # ── Step 1: Grayscale conversion ───────────────────────────────
        # Convert BGR image to grayscale for single-channel processing.
        # CLAHE operates on 2-D intensity images, not colour images.

        # ── Step 2: Apply CLAHE ────────────────────────────────────────
        # clipLimit=2.0  : Maximum contrast amplification per tile.
        #                  Values above this are clipped and redistributed,
        #                  preventing noise amplification.
        # tileGridSize=(8,8): Divides the image into 64 local tiles.
        #                  Each tile gets its own histogram equalization,
        #                  enabling local contrast enhancement.
        # Bilinear interpolation smooths tile boundaries on merge.
        # Grayscale + CLAHE
        gray = cv2.cvtColor(raw_img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        clahe_img = clahe.apply(gray)


        # ── Step 3: Binary thresholding ────────────────────────────────
        # THRESH_BINARY_INV: pixels below threshold=120 become white
        # (aggregate foreground), above become black (background).
        # Threshold of 120 separates the darker aggregate particle
        # from the lighter white background used during imaging.

        # ── Step 4: Morphological opening ──────────────────────────────
        # Erode then dilate with a 3x3 kernel (2 iterations) to remove
        # small isolated noise regions while preserving the main
        # aggregate contour shape.
        # Threshold + cleanup
        _, mask = cv2.threshold(clahe_img, 120, 255, cv2.THRESH_BINARY_INV)
        kernel = np.ones((3, 3), np.uint8)
        clean_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)


        # ── Step 5: Largest contour extraction ─────────────────────────
        # RETR_EXTERNAL retrieves only outermost contours (no holes).
        # Selecting the largest contour by area isolates the single
        # aggregate particle and ignores all background noise blobs.
        # Find largest contour
        contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        # Skip images where no particle contour could be detected
        if not contours:
            print(f"No contour in: {filename}")
            pbar.update(1)
            continue
        # Identify the largest contour — the aggregate particle boundary
        largest = max(contours, key=cv2.contourArea)
        # Create a binary mask by filling the largest contour solid white
        mask_fill = np.zeros_like(gray)
        cv2.drawContours(mask_fill, [largest], -1, 255, thickness=-1)


        # ── Step 6: White background replacement ───────────────────────
        # Pixels inside the particle mask retain their CLAHE-enhanced
        # intensity values. All background pixels are set to 255 (white),
        # producing a clean, consistent background across all images.
        # White background
        white_bg = np.full_like(clahe_img, 255)
        result = np.where(mask_fill == 255, clahe_img, white_bg)


        # ── Step 7: Save processed image ───────────────────────────────
        # Write the CLAHE-enhanced, white-background image to the output
        # folder, preserving the original filename for traceability.
        # Save result
        out_path = os.path.join(output_folder, filename)
        cv2.imwrite(out_path, result)
        print(f"Saved: {filename}")

        # Advance tqdm progress bar by one image
        # Update progress bar
        pbar.update(1)

print(f"\nDone! Processed images from class '500' saved in: {output_folder}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 Found 4084 images in class '500'.


Processing images:   0%|          | 2/4084 [00:00<13:51,  4.91image/s]

✅ Saved: 500_DSC07813_01.png
✅ Saved: 500_DSC07813_02.png


Processing images:   0%|          | 4/4084 [00:00<12:49,  5.30image/s]

✅ Saved: 500_DSC07813_03.png
✅ Saved: 500_DSC07813_04.png


...

✅ Saved: 500_DSC07876_41.png
✅ Saved: 500_DSC07876_42.png
✅ Saved: 500_DSC07876_43.png
✅ Saved: 500_DSC07876_44.png
✅ Saved: 500_DSC07876_45.png
✅ Saved: 500_DSC07876_46.png
✅ Saved: 500_DSC07876_47.png
✅ Saved: 500_DSC07876_48.png
✅ Saved: 500_DSC07876_49.png
✅ Saved: 500_DSC07876_50.png
✅ Saved: 500_DSC07876_51.png
✅ Saved: 500_DSC07876_52.png
✅ Saved: 500_DSC07876_53.png


Processing images: 100%|█████████▉| 4075/4084 [02:55<00:00, 59.25image/s]

✅ Saved: 500_DSC07876_54.png
✅ Saved: 500_DSC07876_55.png
✅ Saved: 500_DSC07876_56.png
✅ Saved: 500_DSC07876_57.png
✅ Saved: 500_DSC07876_58.png
✅ Saved: 500_DSC07876_59.png
✅ Saved: 500_DSC07876_60.png
✅ Saved: 500_DSC07876_61.png
✅ Saved: 500_DSC07876_62.png
✅ Saved: 500_DSC07876_63.png
✅ Saved: 500_DSC07876_64.png
✅ Saved: 500_DSC07876_65.png
✅ Saved: 500_DSC07876_66.png
✅ Saved: 500_DSC07876_67.png


Processing images: 100%|██████████| 4084/4084 [02:55<00:00, 23.27image/s]

✅ Saved: 500_DSC07876_68.png
✅ Saved: 500_DSC07876_69.png
✅ Saved: 500_DSC07876_70.png

🎉 Done! Processed images from class '500' saved in: /content/drive/MyDrive/BG CLAHE/500


In [ ]:
import cv2
import numpy as np
# ── Libraries ─────────────────────────────────────────────────
# cv2      : OpenCV for image processing (CLAHE, thresholding,
#            morphology, contour detection)
# numpy    : Array operations for mask creation and pixel ops
# os/glob  : File system traversal and path construction
# tqdm     : Terminal progress bar for batch processing
import os
from glob import glob
# Mount Google Drive to access input images and save output
from google.colab import drive
from tqdm import tqdm  # Import tqdm for progress bar

# Mount Google Drive
drive.mount('/content/drive')

# === Set the path to your already extracted BG Dataset ===

# ── Paths ──────────────────────────────────────────────────────
# class_folder : Input directory for this milling class
# output_folder: Destination for CLAHE-processed images
class_folder = '/content/drive/MyDrive/BG_Dataset/1000'  # Path to class '0' folder
output_folder = '/content/drive/MyDrive/BG CLAHE/1000'  # Save processed images to BG_CLAHE folder
os.makedirs(output_folder, exist_ok=True)

# === Process all images in class '0' ===

# Discover all image files in the class folder (sorted for
# reproducibility) and wrap in a tqdm progress bar
image_paths = sorted(glob(os.path.join(class_folder, '*')))
print(f"Found {len(image_paths)} images in class '1000'.")

# Initialize tqdm progress bar
with tqdm(total=len(image_paths), desc="Processing images", unit="image") as pbar:
    for path in image_paths:

        # ── Per-image pipeline ─────────────────────────────────────────
        filename = os.path.basename(path)
        raw_img = cv2.imread(path)
        # Skip unreadable files (corrupt or unsupported format)
        if raw_img is None:
            print(f"Skipped: {filename}")
            pbar.update(1)
            continue


        # ── Step 1: Grayscale conversion ───────────────────────────────
        # Convert BGR image to grayscale for single-channel processing.
        # CLAHE operates on 2-D intensity images, not colour images.

        # ── Step 2: Apply CLAHE ────────────────────────────────────────
        # clipLimit=2.0  : Maximum contrast amplification per tile.
        #                  Values above this are clipped and redistributed,
        #                  preventing noise amplification.
        # tileGridSize=(8,8): Divides the image into 64 local tiles.
        #                  Each tile gets its own histogram equalization,
        #                  enabling local contrast enhancement.
        # Bilinear interpolation smooths tile boundaries on merge.
        # Grayscale + CLAHE
        gray = cv2.cvtColor(raw_img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        clahe_img = clahe.apply(gray)


        # ── Step 3: Binary thresholding ────────────────────────────────
        # THRESH_BINARY_INV: pixels below threshold=120 become white
        # (aggregate foreground), above become black (background).
        # Threshold of 120 separates the darker aggregate particle
        # from the lighter white background used during imaging.

        # ── Step 4: Morphological opening ──────────────────────────────
        # Erode then dilate with a 3x3 kernel (2 iterations) to remove
        # small isolated noise regions while preserving the main
        # aggregate contour shape.
        # Threshold + cleanup
        _, mask = cv2.threshold(clahe_img, 120, 255, cv2.THRESH_BINARY_INV)
        kernel = np.ones((3, 3), np.uint8)
        clean_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)


        # ── Step 5: Largest contour extraction ─────────────────────────
        # RETR_EXTERNAL retrieves only outermost contours (no holes).
        # Selecting the largest contour by area isolates the single
        # aggregate particle and ignores all background noise blobs.
        # Find largest contour
        contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        # Skip images where no particle contour could be detected
        if not contours:
            print(f"No contour in: {filename}")
            pbar.update(1)
            continue
        # Identify the largest contour — the aggregate particle boundary
        largest = max(contours, key=cv2.contourArea)
        # Create a binary mask by filling the largest contour solid white
        mask_fill = np.zeros_like(gray)
        cv2.drawContours(mask_fill, [largest], -1, 255, thickness=-1)


        # ── Step 6: White background replacement ───────────────────────
        # Pixels inside the particle mask retain their CLAHE-enhanced
        # intensity values. All background pixels are set to 255 (white),
        # producing a clean, consistent background across all images.
        # White background
        white_bg = np.full_like(clahe_img, 255)
        result = np.where(mask_fill == 255, clahe_img, white_bg)


        # ── Step 7: Save processed image ───────────────────────────────
        # Write the CLAHE-enhanced, white-background image to the output
        # folder, preserving the original filename for traceability.
        # Save result
        out_path = os.path.join(output_folder, filename)
        cv2.imwrite(out_path, result)
        print(f"Saved: {filename}")

        # Advance tqdm progress bar by one image
        # Update progress bar
        pbar.update(1)

print(f"\nDone! Processed images from class '1000' saved in: {output_folder}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 Found 4141 images in class '1000'.


Processing images:   0%|          | 1/4141 [00:00<16:38,  4.14image/s]

✅ Saved: 1000_DSC07969_01.png


Processing images:   0%|          | 2/4141 [00:00<18:12,  3.79image/s]

✅ Saved: 1000_DSC07969_02.png


...

✅ Saved: 1000_DSC08043_42.png
✅ Saved: 1000_DSC08043_43.png
✅ Saved: 1000_DSC08043_44.png
✅ Saved: 1000_DSC08043_45.png
✅ Saved: 1000_DSC08043_46.png
✅ Saved: 1000_DSC08043_47.png
✅ Saved: 1000_DSC08043_48.png
✅ Saved: 1000_DSC08043_49.png
✅ Saved: 1000_DSC08043_50.png
✅ Saved: 1000_DSC08043_51.png
✅ Saved: 1000_DSC08043_52.png
✅ Saved: 1000_DSC08043_53.png
✅ Saved: 1000_DSC08043_54.png


Processing images: 100%|█████████▉| 4134/4141 [03:16<00:00, 53.05image/s]

✅ Saved: 1000_DSC08043_55.png
✅ Saved: 1000_DSC08043_56.png
✅ Saved: 1000_DSC08043_57.png
✅ Saved: 1000_DSC08043_58.png
✅ Saved: 1000_DSC08043_59.png
✅ Saved: 1000_DSC08043_60.png
✅ Saved: 1000_DSC08043_61.png
✅ Saved: 1000_DSC08043_62.png
✅ Saved: 1000_DSC08043_63.png
✅ Saved: 1000_DSC08043_64.png
✅ Saved: 1000_DSC08043_65.png


Processing images: 100%|██████████| 4141/4141 [03:16<00:00, 21.05image/s]

✅ Saved: 1000_DSC08043_66.png
✅ Saved: 1000_DSC08043_67.png
✅ Saved: 1000_DSC08043_68.png
✅ Saved: 1000_DSC08043_69.png
✅ Saved: 1000_DSC08043_70.png

🎉 Done! Processed images from class '1000' saved in: /content/drive/MyDrive/BG CLAHE/1000


In [ ]:
import cv2
import numpy as np
# ── Libraries ─────────────────────────────────────────────────
# cv2      : OpenCV for image processing (CLAHE, thresholding,
#            morphology, contour detection)
# numpy    : Array operations for mask creation and pixel ops
# os/glob  : File system traversal and path construction
# tqdm     : Terminal progress bar for batch processing
import os
from glob import glob
# Mount Google Drive to access input images and save output
from google.colab import drive
from tqdm import tqdm  # Import tqdm for progress bar

# Mount Google Drive
drive.mount('/content/drive')

# === Set the path to your already extracted BG Dataset ===

# ── Paths ──────────────────────────────────────────────────────
# class_folder : Input directory for this milling class
# output_folder: Destination for CLAHE-processed images
class_folder = '/content/drive/MyDrive/BG_Dataset/1500'  # Path to class '0' folder
output_folder = '/content/drive/MyDrive/BG CLAHE/1500'  # Save processed images to BG_CLAHE folder
os.makedirs(output_folder, exist_ok=True)

# === Process all images in class '0' ===

# Discover all image files in the class folder (sorted for
# reproducibility) and wrap in a tqdm progress bar
image_paths = sorted(glob(os.path.join(class_folder, '*')))
print(f"Found {len(image_paths)} images in class '1500'.")

# Initialize tqdm progress bar
with tqdm(total=len(image_paths), desc="Processing images", unit="image") as pbar:
    for path in image_paths:

        # ── Per-image pipeline ─────────────────────────────────────────
        filename = os.path.basename(path)
        raw_img = cv2.imread(path)
        # Skip unreadable files (corrupt or unsupported format)
        if raw_img is None:
            print(f"Skipped: {filename}")
            pbar.update(1)
            continue


        # ── Step 1: Grayscale conversion ───────────────────────────────
        # Convert BGR image to grayscale for single-channel processing.
        # CLAHE operates on 2-D intensity images, not colour images.

        # ── Step 2: Apply CLAHE ────────────────────────────────────────
        # clipLimit=2.0  : Maximum contrast amplification per tile.
        #                  Values above this are clipped and redistributed,
        #                  preventing noise amplification.
        # tileGridSize=(8,8): Divides the image into 64 local tiles.
        #                  Each tile gets its own histogram equalization,
        #                  enabling local contrast enhancement.
        # Bilinear interpolation smooths tile boundaries on merge.
        # Grayscale + CLAHE
        gray = cv2.cvtColor(raw_img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        clahe_img = clahe.apply(gray)


        # ── Step 3: Binary thresholding ────────────────────────────────
        # THRESH_BINARY_INV: pixels below threshold=120 become white
        # (aggregate foreground), above become black (background).
        # Threshold of 120 separates the darker aggregate particle
        # from the lighter white background used during imaging.

        # ── Step 4: Morphological opening ──────────────────────────────
        # Erode then dilate with a 3x3 kernel (2 iterations) to remove
        # small isolated noise regions while preserving the main
        # aggregate contour shape.
        # Threshold + cleanup
        _, mask = cv2.threshold(clahe_img, 120, 255, cv2.THRESH_BINARY_INV)
        kernel = np.ones((3, 3), np.uint8)
        clean_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)


        # ── Step 5: Largest contour extraction ─────────────────────────
        # RETR_EXTERNAL retrieves only outermost contours (no holes).
        # Selecting the largest contour by area isolates the single
        # aggregate particle and ignores all background noise blobs.
        # Find largest contour
        contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        # Skip images where no particle contour could be detected
        if not contours:
            print(f"No contour in: {filename}")
            pbar.update(1)
            continue
        # Identify the largest contour — the aggregate particle boundary
        largest = max(contours, key=cv2.contourArea)
        # Create a binary mask by filling the largest contour solid white
        mask_fill = np.zeros_like(gray)
        cv2.drawContours(mask_fill, [largest], -1, 255, thickness=-1)


        # ── Step 6: White background replacement ───────────────────────
        # Pixels inside the particle mask retain their CLAHE-enhanced
        # intensity values. All background pixels are set to 255 (white),
        # producing a clean, consistent background across all images.
        # White background
        white_bg = np.full_like(clahe_img, 255)
        result = np.where(mask_fill == 255, clahe_img, white_bg)


        # ── Step 7: Save processed image ───────────────────────────────
        # Write the CLAHE-enhanced, white-background image to the output
        # folder, preserving the original filename for traceability.
        # Save result
        out_path = os.path.join(output_folder, filename)
        cv2.imwrite(out_path, result)
        print(f"Saved: {filename}")

        # Advance tqdm progress bar by one image
        # Update progress bar
        pbar.update(1)

print(f"\nDone! Processed images from class '1500' saved in: {output_folder}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 Found 4142 images in class '1500'.


Processing images:   0%|          | 1/4142 [00:00<15:32,  4.44image/s]

✅ Saved: 1500_DSC08057_01.png


Processing images:   0%|          | 2/4142 [00:00<15:33,  4.43image/s]

✅ Saved: 1500_DSC08057_02.png


...

✅ Saved: 1500_DSC08117_45.png
✅ Saved: 1500_DSC08117_46.png
✅ Saved: 1500_DSC08117_47.png
✅ Saved: 1500_DSC08117_48.png
✅ Saved: 1500_DSC08117_49.png
✅ Saved: 1500_DSC08117_50.png
✅ Saved: 1500_DSC08117_51.png
✅ Saved: 1500_DSC08117_52.png
✅ Saved: 1500_DSC08117_53.png
✅ Saved: 1500_DSC08117_54.png
✅ Saved: 1500_DSC08117_55.png


Processing images: 100%|█████████▉| 4137/4142 [04:00<00:00, 57.31image/s]

✅ Saved: 1500_DSC08117_56.png
✅ Saved: 1500_DSC08117_57.png
✅ Saved: 1500_DSC08117_58.png
✅ Saved: 1500_DSC08117_59.png
✅ Saved: 1500_DSC08117_60.png
✅ Saved: 1500_DSC08117_61.png
✅ Saved: 1500_DSC08117_62.png
✅ Saved: 1500_DSC08117_63.png
✅ Saved: 1500_DSC08117_64.png
✅ Saved: 1500_DSC08117_65.png
✅ Saved: 1500_DSC08117_66.png
✅ Saved: 1500_DSC08117_67.png
✅ Saved: 1500_DSC08117_68.png
✅ Saved: 1500_DSC08117_69.png


Processing images: 100%|██████████| 4142/4142 [04:00<00:00, 17.23image/s]

✅ Saved: 1500_DSC08117_70.png

🎉 Done! Processed images from class '1500' saved in: /content/drive/MyDrive/BG CLAHE/1500


In [ ]:
import cv2
import numpy as np
# ── Libraries ─────────────────────────────────────────────────
# cv2      : OpenCV for image processing (CLAHE, thresholding,
#            morphology, contour detection)
# numpy    : Array operations for mask creation and pixel ops
# os/glob  : File system traversal and path construction
# tqdm     : Terminal progress bar for batch processing
import os
from glob import glob
# Mount Google Drive to access input images and save output
from google.colab import drive
from tqdm import tqdm  # Import tqdm for progress bar

# Mount Google Drive
drive.mount('/content/drive')

# === Set the path to your already extracted BG Dataset ===

# ── Paths ──────────────────────────────────────────────────────
# class_folder : Input directory for this milling class
# output_folder: Destination for CLAHE-processed images
class_folder = '/content/drive/MyDrive/BG_Dataset/2000'  # Path to class '0' folder
output_folder = '/content/drive/MyDrive/BG CLAHE/2000'  # Save processed images to BG_CLAHE folder
os.makedirs(output_folder, exist_ok=True)

# === Process all images in class '0' ===

# Discover all image files in the class folder (sorted for
# reproducibility) and wrap in a tqdm progress bar
image_paths = sorted(glob(os.path.join(class_folder, '*')))
print(f"Found {len(image_paths)} images in class '2000'.")

# Initialize tqdm progress bar
with tqdm(total=len(image_paths), desc="Processing images", unit="image") as pbar:
    for path in image_paths:

        # ── Per-image pipeline ─────────────────────────────────────────
        filename = os.path.basename(path)
        raw_img = cv2.imread(path)
        # Skip unreadable files (corrupt or unsupported format)
        if raw_img is None:
            print(f"Skipped: {filename}")
            pbar.update(1)
            continue


        # ── Step 1: Grayscale conversion ───────────────────────────────
        # Convert BGR image to grayscale for single-channel processing.
        # CLAHE operates on 2-D intensity images, not colour images.

        # ── Step 2: Apply CLAHE ────────────────────────────────────────
        # clipLimit=2.0  : Maximum contrast amplification per tile.
        #                  Values above this are clipped and redistributed,
        #                  preventing noise amplification.
        # tileGridSize=(8,8): Divides the image into 64 local tiles.
        #                  Each tile gets its own histogram equalization,
        #                  enabling local contrast enhancement.
        # Bilinear interpolation smooths tile boundaries on merge.
        # Grayscale + CLAHE
        gray = cv2.cvtColor(raw_img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        clahe_img = clahe.apply(gray)


        # ── Step 3: Binary thresholding ────────────────────────────────
        # THRESH_BINARY_INV: pixels below threshold=120 become white
        # (aggregate foreground), above become black (background).
        # Threshold of 120 separates the darker aggregate particle
        # from the lighter white background used during imaging.

        # ── Step 4: Morphological opening ──────────────────────────────
        # Erode then dilate with a 3x3 kernel (2 iterations) to remove
        # small isolated noise regions while preserving the main
        # aggregate contour shape.
        # Threshold + cleanup
        _, mask = cv2.threshold(clahe_img, 120, 255, cv2.THRESH_BINARY_INV)
        kernel = np.ones((3, 3), np.uint8)
        clean_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)


        # ── Step 5: Largest contour extraction ─────────────────────────
        # RETR_EXTERNAL retrieves only outermost contours (no holes).
        # Selecting the largest contour by area isolates the single
        # aggregate particle and ignores all background noise blobs.
        # Find largest contour
        contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        # Skip images where no particle contour could be detected
        if not contours:
            print(f"No contour in: {filename}")
            pbar.update(1)
            continue
        # Identify the largest contour — the aggregate particle boundary
        largest = max(contours, key=cv2.contourArea)
        # Create a binary mask by filling the largest contour solid white
        mask_fill = np.zeros_like(gray)
        cv2.drawContours(mask_fill, [largest], -1, 255, thickness=-1)


        # ── Step 6: White background replacement ───────────────────────
        # Pixels inside the particle mask retain their CLAHE-enhanced
        # intensity values. All background pixels are set to 255 (white),
        # producing a clean, consistent background across all images.
        # White background
        white_bg = np.full_like(clahe_img, 255)
        result = np.where(mask_fill == 255, clahe_img, white_bg)


        # ── Step 7: Save processed image ───────────────────────────────
        # Write the CLAHE-enhanced, white-background image to the output
        # folder, preserving the original filename for traceability.
        # Save result
        out_path = os.path.join(output_folder, filename)
        cv2.imwrite(out_path, result)
        print(f"Saved: {filename}")

        # Advance tqdm progress bar by one image
        # Update progress bar
        pbar.update(1)

print(f"\nDone! Processed images from class '2000' saved in: {output_folder}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 Found 4180 images in class '2000'.


Processing images:   0%|          | 2/4180 [00:00<15:01,  4.64image/s]

✅ Saved: 2000_2000 Rev (1)_07.png
✅ Saved: 2000_2000 Rev (1)_08.png


Processing images:   0%|          | 3/4180 [00:00<15:21,  4.53image/s]

✅ Saved: 2000_2000 Rev (1)_09.png


...

✅ Saved: 2000_2000 Rev (9)_33.png
✅ Saved: 2000_2000 Rev (9)_34.png
✅ Saved: 2000_2000 Rev (9)_35.png
✅ Saved: 2000_2000 Rev (9)_36.png
✅ Saved: 2000_2000 Rev (9)_37.png
✅ Saved: 2000_2000 Rev (9)_38.png
✅ Saved: 2000_2000 Rev (9)_39.png
✅ Saved: 2000_2000 Rev (9)_40.png
✅ Saved: 2000_2000 Rev (9)_41.png
✅ Saved: 2000_2000 Rev (9)_42.png
✅ Saved: 2000_2000 Rev (9)_43.png
✅ Saved: 2000_2000 Rev (9)_44.png
✅ Saved: 2000_2000 Rev (9)_45.png


Processing images: 100%|█████████▉| 4168/4180 [03:19<00:00, 61.23image/s]

✅ Saved: 2000_2000 Rev (9)_46.png
✅ Saved: 2000_2000 Rev (9)_47.png
✅ Saved: 2000_2000 Rev (9)_48.png
✅ Saved: 2000_2000 Rev (9)_49.png
✅ Saved: 2000_2000 Rev (9)_50.png
✅ Saved: 2000_2000 Rev (9)_51.png
✅ Saved: 2000_2000 Rev (9)_52.png
✅ Saved: 2000_2000 Rev (9)_53.png
✅ Saved: 2000_2000 Rev (9)_54.png
✅ Saved: 2000_2000 Rev (9)_55.png
✅ Saved: 2000_2000 Rev (9)_56.png
✅ Saved: 2000_2000 Rev (9)_57.png
✅ Saved: 2000_2000 Rev (9)_58.png


Processing images: 100%|██████████| 4180/4180 [03:19<00:00, 20.93image/s]

✅ Saved: 2000_2000 Rev (9)_59.png
✅ Saved: 2000_2000 Rev (9)_60.png
✅ Saved: 2000_2000 Rev (9)_61.png
✅ Saved: 2000_2000 Rev (9)_62.png
✅ Saved: 2000_2000 Rev (9)_63.png
✅ Saved: 2000_2000 Rev (9)_64.png
✅ Saved: 2000_2000 Rev (9)_65.png
✅ Saved: 2000_2000 Rev (9)_66.png
✅ Saved: 2000_2000 Rev (9)_67.png
✅ Saved: 2000_2000 Rev (9)_68.png
✅ Saved: 2000_2000 Rev (9)_69.png
✅ Saved: 2000_2000 Rev (9)_70.png

🎉 Done! Processed images from class '2000' saved in: /content/drive/MyDrive/BG CLAHE/2000


In [ ]:
import cv2
import numpy as np
# ── Libraries ─────────────────────────────────────────────────
# cv2      : OpenCV for image processing (CLAHE, thresholding,
#            morphology, contour detection)
# numpy    : Array operations for mask creation and pixel ops
# os/glob  : File system traversal and path construction
# tqdm     : Terminal progress bar for batch processing
import os
from glob import glob
# Mount Google Drive to access input images and save output
from google.colab import drive
from tqdm import tqdm  # Import tqdm for progress bar

# Mount Google Drive
drive.mount('/content/drive')

# === Set the path to your already extracted BG Dataset ===

# ── Paths ──────────────────────────────────────────────────────
# class_folder : Input directory for this milling class
# output_folder: Destination for CLAHE-processed images
class_folder = '/content/drive/MyDrive/folder_agg/100_cropped)'  # Path to class '0' folder
output_folder = '/content/drive/MyDrive/NBG CLAHE/100'  # Save processed images to BG_CLAHE folder
os.makedirs(output_folder, exist_ok=True)

# === Process all images in class '0' ===

# Discover all image files in the class folder (sorted for
# reproducibility) and wrap in a tqdm progress bar
image_paths = sorted(glob(os.path.join(class_folder, '*')))
print(f"Found {len(image_paths)} images in class '100'.")

# Initialize tqdm progress bar
with tqdm(total=len(image_paths), desc="Processing images", unit="image") as pbar:
    for path in image_paths:

        # ── Per-image pipeline ─────────────────────────────────────────
        filename = os.path.basename(path)
        raw_img = cv2.imread(path)
        # Skip unreadable files (corrupt or unsupported format)
        if raw_img is None:
            print(f"Skipped: {filename}")
            pbar.update(1)
            continue


        # ── Step 1: Grayscale conversion ───────────────────────────────
        # Convert BGR image to grayscale for single-channel processing.
        # CLAHE operates on 2-D intensity images, not colour images.

        # ── Step 2: Apply CLAHE ────────────────────────────────────────
        # clipLimit=2.0  : Maximum contrast amplification per tile.
        #                  Values above this are clipped and redistributed,
        #                  preventing noise amplification.
        # tileGridSize=(8,8): Divides the image into 64 local tiles.
        #                  Each tile gets its own histogram equalization,
        #                  enabling local contrast enhancement.
        # Bilinear interpolation smooths tile boundaries on merge.
        # Grayscale + CLAHE
        gray = cv2.cvtColor(raw_img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        clahe_img = clahe.apply(gray)


        # ── Step 3: Binary thresholding ────────────────────────────────
        # THRESH_BINARY_INV: pixels below threshold=120 become white
        # (aggregate foreground), above become black (background).
        # Threshold of 120 separates the darker aggregate particle
        # from the lighter white background used during imaging.

        # ── Step 4: Morphological opening ──────────────────────────────
        # Erode then dilate with a 3x3 kernel (2 iterations) to remove
        # small isolated noise regions while preserving the main
        # aggregate contour shape.
        # Threshold + cleanup
        _, mask = cv2.threshold(clahe_img, 120, 255, cv2.THRESH_BINARY_INV)
        kernel = np.ones((3, 3), np.uint8)
        clean_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)


        # ── Step 5: Largest contour extraction ─────────────────────────
        # RETR_EXTERNAL retrieves only outermost contours (no holes).
        # Selecting the largest contour by area isolates the single
        # aggregate particle and ignores all background noise blobs.
        # Find largest contour
        contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        # Skip images where no particle contour could be detected
        if not contours:
            print(f"No contour in: {filename}")
            pbar.update(1)
            continue
        # Identify the largest contour — the aggregate particle boundary
        largest = max(contours, key=cv2.contourArea)
        # Create a binary mask by filling the largest contour solid white
        mask_fill = np.zeros_like(gray)
        cv2.drawContours(mask_fill, [largest], -1, 255, thickness=-1)


        # ── Step 6: White background replacement ───────────────────────
        # Pixels inside the particle mask retain their CLAHE-enhanced
        # intensity values. All background pixels are set to 255 (white),
        # producing a clean, consistent background across all images.
        # White background
        white_bg = np.full_like(clahe_img, 255)
        result = np.where(mask_fill == 255, clahe_img, white_bg)


        # ── Step 7: Save processed image ───────────────────────────────
        # Write the CLAHE-enhanced, white-background image to the output
        # folder, preserving the original filename for traceability.
        # Save result
        out_path = os.path.join(output_folder, filename)
        cv2.imwrite(out_path, result)
        print(f"Saved: {filename}")

        # Advance tqdm progress bar by one image
        # Update progress bar
        pbar.update(1)

print(f"\nDone! Processed images from class '100' saved in: {output_folder}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 Found 0 images in class '100'.


Processing images: 0image [00:00, ?image/s]


🎉 Done! Processed images from class '100' saved in: /content/drive/MyDrive/NBG CLAHE/100
